# Самостоятельная работа: билеты на мероприятия

Вы аналитик билетной платформы. Пришла выгрузка продаж за март-май 2026 года: файл `concert_tickets.csv`, ~2 500 строк.

**Задача от руководителя:**
1. Выручка по типам мероприятий (концерты, футбол, стендап) - по убыванию
2. Топ-5 артистов/событий по среднему рейтингу зрителей

В данных не менее **десяти проблем**. Какие именно - не сказано.
Найдите командами разведки, запишите в журнал, почистите, посчитайте сводки.

In [8]:
import numpy as np
import pandas as pd

## Шпаргалка

**Разведка**

| Команда | Что показывает |
|---|---|
| `df.shape`, `df.head()` | размер и первые строки |
| `df.info()` | типы и число непустых значений |
| `df.describe()` | min, max, среднее, квартили |
| `df.isnull().sum()` | пропуски по столбцам |
| `df.duplicated().sum()` | полные дубликаты строк |
| `s.value_counts(dropna=False)` | частоты значений включая NaN |
| `s.nunique()`, `s.min()`, `s.max()` | уникальных, мин, макс |

**Чистка**

| Команда | Что делает |
|---|---|
| `df.columns.str.strip().str.lower().str.replace(" ", "_")` | имена столбцов |
| `s.str.strip()`, `s.str.title()`, `s.str.lower()` | пробелы и регистр |
| `pd.to_numeric(s, errors="coerce")` | текст в число |
| `pd.to_datetime(s, errors="coerce")` | текст в дату |
| `s.fillna(x)` | заполнить пропуски |
| `df.drop_duplicates()` | удалить повторы |
| `df.loc[mask, "col"] = x` | замена по условию |
| `df[df["col"] > 0]` | отфильтровать строки |


**Regex: методы pandas**

| Метод | Что делает |
|---|---|
| `s.str.replace(r"pat", "замена", regex=True)` | заменить совпадение |
| `s.str.contains(r"pat", regex=True)` | возвращает True/False |
| `s.str.extract(r"(группа)")` | извлечь первую группу `()` |
| `s.str.findall(r"pat")` | список всех совпадений |
| `s.str.count(r"pat")` | сколько раз встречается |

**Regex: паттерны**

| Паттерн | Что значит |
|---|---|
| `[ ,]` | символ-класс: пробел ИЛИ запятая |
| `[^\d]` | всё кроме цифр |
| `\d` | любая цифра 0-9 |
| `\D` | всё что не цифра |
| `\s` | пробел, таб, перенос строки |
| `\w` | буква, цифра или `_` |
| `+` | один или более |
| `*` | ноль или более |
| `^` внутри `[]` | исключение из класса |
| `r"..."` | raw string: `\d` не трактуется Python |

**Сводки**

| Команда | Что делает |
|---|---|
| `df.groupby("col")["x"].sum()` | сумма по группам |
| `df.groupby("col")["x"].mean()` | среднее по группам |
| `.sort_values(ascending=False).head(5)` | топ-5 |

## Работа

In [9]:
df = pd.read_csv("../data/concert_tickets.csv")
df_raw = df.copy()

In [11]:
# checking the columns
df.columns

Index([' Ticket ID', 'Artist / Event ', 'Event Type', 'Venue', 'Event Date',
       'Purchase Date', 'Customer Region', 'Ticket Type', 'Price (UZS)',
       'Quantity', ' Payment', 'Status', 'Rating'],
      dtype='str')

In [12]:
# checking duplicates
df.duplicated().value_counts()

False    2501
True       39
Name: count, dtype: int64

In [13]:
# removing dulpicates
df = df.drop_duplicates()

In [14]:
# columns cleaning
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)

In [15]:
# checking the cleaned state
df.columns

Index(['ticket_id', 'artist_event', 'event_type', 'venue', 'event_date',
       'purchase_date', 'customer_region', 'ticket_type', 'price_uzs',
       'quantity', 'payment', 'status', 'rating'],
      dtype='str')

In [16]:
# view of overall table of first 10 rows
df.head(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
0,TK-11557,Jasur Umirov,Konsert,Uzbekiston Konsert Zali,2026-04-12,2026-02-22,Farg'ona,VIP,454840,3,naqd,YAKUNLANGAN,4.8
1,TK-10134,Doniyor Otajonov,Konsert,Uzbekiston Konsert Zali,03.05.2026,2026-04-12,Farg'ona,Standart,144969,4,karta,yakunlangan,3.7
2,TK-11639,Doniyor Otajonov,konsert,Uzbekiston Konsert Zali,03.05.2026,2026-03-25,Qashqadaryo,Standart,161021,1,qr,yakunlangan,4.5
3,TK-11498,Sevinch Mominova,Konsert,Humo Arena,2026-03-15,2026-01-29,Xorazm,VIP,477421,1,karta,yakunlangan,4.0
4,TK-11399,Pakhtakor vs Lokomotiv,Futbol,Milliy Stadion,10.05.2026,2026-03-31,Namangan,Standart,130849,2,karta,yakunlangan,4.7
5,TK-10471,Jasur Umirov,Konsert,Uzbekiston Konsert Zali,12.04.2026,2026-02-24,Toshkent,Fan,102771,2,qr,qaytarilgan,NaN
6,TK-11513,Doniyor Otajonov,Konsert,Uzbekiston Konsert Zali,2026-05-03,2026-04-26,Andijon,VIP,451 283 so'm,2,naqd,KUTILMOQDA,NaN
7,TK-12274,Pakhtakor vs Bunyodkor,Futbol,Milliy Stadion,2026-04-05,2026-03-10,Qashqadaryo,Fan,93002,3,qr,yakunlangan,4.6
8,TK-10044,Pakhtakor vs Lokomotiv,Futbol,Milliy Stadion,10.05.2026,2026-03-26,Samarqand,Fan,80956,1,qr,yakunlangan,4.9
9,TK-11391,Doniyor Otajonov,Konsert,Uzbekiston Konsert Zali,03.05.2026,2026-03-11,Namangan,Standart,159 184 so'm,1,qr,KUTILMOQDA,NaN


In [17]:
# checking are there no duplicates in different ways
df["artist_event"].unique()

<StringArray>
[          'Jasur Umirov',       'Doniyor Otajonov',       'Sevinch Mominova',
 'Pakhtakor vs Lokomotiv', 'Pakhtakor vs Bunyodkor',       'Navruz Gala Shou',
 'Ulug'bek Rahmatullayev',       'Shaxlo Ismoilova',      'QXL Stand-Up Show',
     'Muhabbat Shamayeva',       'SEVINCH MOMINOVA',       'sevinch mominova',
      'Sevinch Mo'minova']
Length: 13, dtype: str

In [18]:
# checking are there no duplicates in different ways
df["ticket_id"].unique()

<StringArray>
['TK-11557', 'TK-10134', 'TK-11639', 'TK-11498', 'TK-11399', 'TK-10471',
 'TK-11513', 'TK-12274', 'TK-10044', 'TK-11391',
 ...
 'TK-11482', 'TK-10330', 'TK-11238', 'TK-10466', 'TK-12169', 'TK-11638',
 'TK-11095', 'TK-11130', 'TK-11294', 'TK-10860']
Length: 2500, dtype: str

In [19]:
# checking are there no duplicates in different ways
df["event_type"].unique()

<StringArray>
[ 'Konsert',  'konsert',   'Futbol', 'Konsert ',  'StendAp',  'KONSERT',
   'FUTBOL',   'futbol']
Length: 8, dtype: str

In [20]:
# removing duplicates in ticke ID
df["ticket_id"].drop_duplicates()

0       TK-11557
1       TK-10134
2       TK-11639
3       TK-11498
4       TK-11399
          ...   
2535    TK-11638
2536    TK-11095
2537    TK-11130
2538    TK-11294
2539    TK-10860
Name: ticket_id, Length: 2500, dtype: str

In [21]:
# fixing artist_evennt column
df["artist_event"] = df["artist_event"].str.strip().str.lower().str.replace("'", "", regex=False)

In [22]:
# checking the uniqueness
df["artist_event"].unique()

<StringArray>
[          'jasur umirov',       'doniyor otajonov',       'sevinch mominova',
 'pakhtakor vs lokomotiv', 'pakhtakor vs bunyodkor',       'navruz gala shou',
  'ulugbek rahmatullayev',       'shaxlo ismoilova',      'qxl stand-up show',
     'muhabbat shamayeva']
Length: 10, dtype: str

In [23]:
# view of table
df.sample(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
1342,TK-11793,shaxlo ismoilova,Konsert,Humo Arena,19.04.2026,2026-03-22,Farg'ona,Standart,168 072 so'm,3,naqd,qaytarilgan,NaN
35,TK-11393,muhabbat shamayeva,Konsert,Uzbekiston Konsert Zali,2026-05-17,2026-03-19,Qashqadaryo,Fan,87145,2,karta,YAKUNLANGAN,3.9
1403,TK-11002,muhabbat shamayeva,Konsert,Uzbekiston Konsert Zali,2026-05-17,2026-04-27,Buxoro,Fan,95 984 so'm,3,karta,yakunlangan,4.9
758,TK-11716,qxl stand-up show,StendAp,Hysteria Club,26.04.2026,2026-04-13,Namangan,Standart,179642,3,qr,yakunlangan,4.6
1059,TK-11344,doniyor otajonov,Konsert,Uzbekiston Konsert Zali,03.05.2026,2026-04-05,Namangan,VIP,456492,1,qr,yakunlangan,3.8
1735,TK-10718,shaxlo ismoilova,Konsert,Humo Arena,2026-04-19,2026-04-12,Qashqadaryo,VIP,454565,2,qr,YAKUNLANGAN,4.6
1086,TK-11070,sevinch mominova,Konsert,Humo Arena,2026-03-15,2026-03-05,Toshkent,VIP,445183,1,naqd,YAKUNLANGAN,4.1
1641,TK-10919,shaxlo ismoilova,Konsert,Humo Arena,19.04.2026,2026-04-10,Namangan,VIP,430307,2,naqd,YAKUNLANGAN,3.7
98,TK-11760,pakhtakor vs bunyodkor,Futbol,Milliy Stadion,2026-04-05,2026-03-22,Andijon,VIP,451494,2,qr,yakunlangan,4.6
1979,TK-11035,jasur umirov,Konsert,Uzbekiston Konsert Zali,12.04.2026,2026-04-11,Farg'ona,VIP,463 192 so'm,2,karta,YAKUNLANGAN,4.6


In [24]:
# cleaning event_type
df["event_type"] = df["event_type"].str.strip().str.lower()

In [25]:
# checking are there no duplicates in different ways
df["event_type"].unique()

<StringArray>
['konsert', 'futbol', 'stendap']
Length: 3, dtype: str

In [26]:
# view after checking
df.head(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
0,TK-11557,jasur umirov,konsert,Uzbekiston Konsert Zali,2026-04-12,2026-02-22,Farg'ona,VIP,454840,3,naqd,YAKUNLANGAN,4.8
1,TK-10134,doniyor otajonov,konsert,Uzbekiston Konsert Zali,03.05.2026,2026-04-12,Farg'ona,Standart,144969,4,karta,yakunlangan,3.7
2,TK-11639,doniyor otajonov,konsert,Uzbekiston Konsert Zali,03.05.2026,2026-03-25,Qashqadaryo,Standart,161021,1,qr,yakunlangan,4.5
3,TK-11498,sevinch mominova,konsert,Humo Arena,2026-03-15,2026-01-29,Xorazm,VIP,477421,1,karta,yakunlangan,4.0
4,TK-11399,pakhtakor vs lokomotiv,futbol,Milliy Stadion,10.05.2026,2026-03-31,Namangan,Standart,130849,2,karta,yakunlangan,4.7
5,TK-10471,jasur umirov,konsert,Uzbekiston Konsert Zali,12.04.2026,2026-02-24,Toshkent,Fan,102771,2,qr,qaytarilgan,NaN
6,TK-11513,doniyor otajonov,konsert,Uzbekiston Konsert Zali,2026-05-03,2026-04-26,Andijon,VIP,451 283 so'm,2,naqd,KUTILMOQDA,NaN
7,TK-12274,pakhtakor vs bunyodkor,futbol,Milliy Stadion,2026-04-05,2026-03-10,Qashqadaryo,Fan,93002,3,qr,yakunlangan,4.6
8,TK-10044,pakhtakor vs lokomotiv,futbol,Milliy Stadion,10.05.2026,2026-03-26,Samarqand,Fan,80956,1,qr,yakunlangan,4.9
9,TK-11391,doniyor otajonov,konsert,Uzbekiston Konsert Zali,03.05.2026,2026-03-11,Namangan,Standart,159 184 so'm,1,qr,KUTILMOQDA,NaN


In [27]:
# checking are there no duplicates in different ways
df["venue"].unique()

<StringArray>
['Uzbekiston Konsert Zali', 'Humo Arena', 'Milliy Stadion', 'Hysteria Club']
Length: 4, dtype: str

In [28]:
# cleaning
df["venue"] = df["venue"].str.strip().str.lower()

In [29]:
# checking
df["venue"]

0       uzbekiston konsert zali
1       uzbekiston konsert zali
2       uzbekiston konsert zali
3                    humo arena
4                milliy stadion
                 ...           
2535                 humo arena
2536             milliy stadion
2537                 humo arena
2538             milliy stadion
2539                 humo arena
Name: venue, Length: 2501, dtype: str

In [30]:
# view after cleaning and check
df.sample(5)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
196,TK-12273,shaxlo ismoilova,konsert,humo arena,2026-04-19,2026-04-14,Qashqadaryo,Fan,105195,2,naqd,kutilmoqda,NaN
2264,TK-10546,ulugbek rahmatullayev,konsert,humo arena,2026-03-22,2026-03-03,Andijon,Standart,156452,3,karta,YAKUNLANGAN,3.6
150,TK-10821,navruz gala shou,konsert,humo arena,2026-03-21,2026-02-15,Qashqadaryo,VIP,459478,4,qr,yakunlangan,3.6
1051,TK-12286,sevinch mominova,konsert,humo arena,15.03.2026,2026-02-06,Xorazm,VIP,459240,4,karta,YAKUNLANGAN,4.2
814,TK-10672,pakhtakor vs bunyodkor,futbol,milliy stadion,05.04.2026,2026-03-15,Qashqadaryo,VIP,441415,3,naqd,YAKUNLANGAN,4.0


In [31]:
df["purchase_date"].value_counts()

purchase_date
2026-03-19    44
2026-03-18    42
2026-03-06    40
2026-04-04    39
2026-03-22    39
              ..
2026-05-10     2
2026-05-16     2
2026-01-14     2
2026-05-13     1
2026-01-19     1
Name: count, Length: 122, dtype: int64

In [32]:
df["purchase_date"].duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
2535     True
2536     True
2537     True
2538     True
2539     True
Name: purchase_date, Length: 2501, dtype: bool

In [33]:
df["customer_region"] = df["customer_region"].str.strip().str.lower()

In [34]:
df["customer_region"].unique()

<StringArray>
[   'farg'ona', 'qashqadaryo',      'xorazm',    'namangan',    'toshkent',
     'andijon',   'samarqand',      'buxoro']
Length: 8, dtype: str

In [35]:
df["ticket_type"].value_counts()

ticket_type
VIP         839
Standart    836
Fan         826
Name: count, dtype: int64

In [36]:
df["ticket_type"] = df["ticket_type"].str.strip().str.lower()

In [37]:
df["ticket_type"].unique()

<StringArray>
['vip', 'standart', 'fan']
Length: 3, dtype: str

In [38]:
df.sample(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
1083,TK-11043,jasur umirov,konsert,uzbekiston konsert zali,2026-04-12,2026-03-23,xorazm,vip,474278,1,qr,QAYTARILGAN,NaN
210,TK-11979,qxl stand-up show,stendap,hysteria club,2026-04-26,2026-03-25,qashqadaryo,standart,138156,1,qr,yakunlangan,4.9
660,TK-10902,navruz gala shou,konsert,humo arena,21.03.2026,2026-02-24,xorazm,standart,157245,2,naqd,kutilmoqda,NaN
1436,TK-10935,shaxlo ismoilova,konsert,humo arena,2026-04-19,2026-03-20,samarqand,vip,472464,1,naqd,YAKUNLANGAN,3.9
1892,TK-11460,qxl stand-up show,stendap,hysteria club,2026-04-26,2026-03-17,xorazm,standart,175 575 so'm,3,qr,kutilmoqda,NaN
1287,TK-12245,doniyor otajonov,konsert,uzbekiston konsert zali,03.05.2026,2026-04-21,farg'ona,vip,435 150 so'm,3,qr,QAYTARILGAN,NaN
48,TK-10296,sevinch mominova,konsert,humo arena,2026-03-15,2026-02-05,xorazm,vip,454897,3,naqd,qaytarilgan,NaN
1463,TK-10355,navruz gala shou,konsert,humo arena,2026-03-21,2026-02-17,andijon,fan,76122,1,naqd,qaytarilgan,NaN
1706,TK-12263,jasur umirov,konsert,uzbekiston konsert zali,2026-04-12,2026-03-17,qashqadaryo,fan,85532,1,qr,yakunlangan,3.9
2181,TK-12379,shaxlo ismoilova,konsert,humo arena,19.04.2026,2026-04-05,andijon,vip,478 862 so'm,1,naqd,yakunlangan,3.6


In [39]:
df["price_uzs"] = (
    df["price_uzs"]
    .astype("string")
    .str.lower()
    .str.replace("so'm", "", regex=False)
    .str.replace("som", "", regex=False)
    .str.replace("uzs", "", regex=False)
    .str.replace(r"[^\d.-]", "", regex=True)
)

# negative and zero price is not cleaned

In [40]:
df.loc[df["price_uzs"] == 0, "price_uzs"] = np.nan

In [41]:
df["price_uzs"].value_counts()

price_uzs
0          10
-150000     3
85532       2
101004      2
459478      2
           ..
443014      1
164075      1
463372      1
86315       1
87538       1
Name: count, Length: 2470, dtype: Int64

In [42]:
df["price_uzs"].value_counts()

price_uzs
0          10
-150000     3
85532       2
101004      2
459478      2
           ..
443014      1
164075      1
463372      1
86315       1
87538       1
Name: count, Length: 2470, dtype: Int64

In [43]:
df.sample(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
1413,TK-11712,pakhtakor vs lokomotiv,futbol,milliy stadion,10.05.2026,2026-03-31,farg'ona,fan,103683,1,karta,yakunlangan,3.5
1419,TK-10346,muhabbat shamayeva,konsert,uzbekiston konsert zali,2026-05-17,2026-03-22,andijon,vip,454960,1,naqd,kutilmoqda,NaN
2424,TK-11881,navruz gala shou,konsert,humo arena,21.03.2026,2026-02-09,namangan,vip,476689,4,naqd,qaytarilgan,NaN
2210,TK-11148,navruz gala shou,konsert,humo arena,2026-03-21,2026-01-30,andijon,vip,452666,4,karta,QAYTARILGAN,NaN
1918,TK-11262,muhabbat shamayeva,konsert,uzbekiston konsert zali,17.05.2026,2026-04-11,farg'ona,standart,169694,1,naqd,yakunlangan,4.7
914,TK-11127,sevinch mominova,konsert,humo arena,2026-03-15,2026-02-24,toshkent,fan,92026,1,qr,yakunlangan,3.9
2310,TK-12101,qxl stand-up show,stendap,hysteria club,2026-04-26,2026-04-06,andijon,fan,83861,4,karta,yakunlangan,3.9
1034,TK-11547,navruz gala shou,konsert,humo arena,2026-03-21,2026-02-23,samarqand,fan,100401,2,karta,yakunlangan,3.6
1839,TK-10102,muhabbat shamayeva,konsert,uzbekiston konsert zali,2026-05-17,2026-04-04,farg'ona,vip,430536,3,karta,YAKUNLANGAN,3.7
1789,TK-11868,navruz gala shou,konsert,humo arena,21.03.2026,2026-03-09,xorazm,vip,454809,4,naqd,yakunlangan,4.7


In [44]:
df["status"] = df["status"].str.strip().str.lower()

In [45]:
df["status"].unique()

<StringArray>
['yakunlangan', 'qaytarilgan', 'kutilmoqda']
Length: 3, dtype: str

In [46]:
df["status"].sample(10)

937      kutilmoqda
864     yakunlangan
223     yakunlangan
2511    qaytarilgan
640     yakunlangan
2229    qaytarilgan
78      yakunlangan
634      kutilmoqda
301     yakunlangan
1548    yakunlangan
Name: status, dtype: str

In [47]:
df.sample(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating
177,TK-11847,pakhtakor vs lokomotiv,futbol,milliy stadion,2026-05-10,2026-04-24,farg'ona,standart,162568,2,karta,yakunlangan,4.8
28,TK-10414,doniyor otajonov,konsert,uzbekiston konsert zali,03.05.2026,2026-05-02,buxoro,vip,450670,2,qr,yakunlangan,5.0
448,TK-11114,pakhtakor vs bunyodkor,futbol,milliy stadion,05.04.2026,2026-03-07,farg'ona,fan,96722,4,qr,yakunlangan,4.7
1621,TK-11458,shaxlo ismoilova,konsert,humo arena,2026-04-19,2026-04-03,namangan,vip,461626,3,naqd,kutilmoqda,NaN
812,TK-12107,muhabbat shamayeva,konsert,uzbekiston konsert zali,2026-05-17,2026-04-27,andijon,standart,136040,4,naqd,kutilmoqda,NaN
1147,TK-10981,pakhtakor vs lokomotiv,futbol,milliy stadion,10.05.2026,2026-05-02,buxoro,vip,444896,3,karta,yakunlangan,4.6
985,TK-11174,pakhtakor vs bunyodkor,futbol,milliy stadion,2026-04-05,2026-04-03,xorazm,vip,465339,3,naqd,yakunlangan,4.1
880,TK-10590,pakhtakor vs lokomotiv,futbol,milliy stadion,2026-05-10,2026-04-17,farg'ona,vip,439334,1,qr,yakunlangan,3.9
2397,TK-11408,navruz gala shou,konsert,humo arena,2026-03-21,2026-01-31,toshkent,standart,177970,3,naqd,yakunlangan,3.8
1103,TK-11210,shaxlo ismoilova,konsert,humo arena,2026-04-19,2026-03-26,samarqand,fan,103461,4,naqd,yakunlangan,3.8


## Итоговые сводки

In [48]:
# Top 5 artists by Rating

df.groupby("artist_event")["rating"].mean().sort_values(ascending=False).head(5)

artist_event
muhabbat shamayeva       4.276111
ulugbek rahmatullayev    4.273934
doniyor otajonov         4.264322
jasur umirov             4.258289
navruz gala shou         4.248913
Name: rating, dtype: float64

In [49]:
df["price_uzs"] = pd.to_numeric(df["price_uzs"], errors="coerce")

In [50]:
df["revenue"] = df["price_uzs"] * df["quantity"]

In [51]:
df["revenue"]

0       1364520
1        579876
2        161021
3        477421
4        261698
         ...   
2535    1772056
2536     164075
2537    1390116
2538      86315
2539     262614
Name: revenue, Length: 2501, dtype: Int64

In [52]:
# Revenue of Events

df.groupby("event_type")["revenue"].sum().sort_values(ascending=False).head()

event_type
konsert    945782973
futbol     291647940
stendap    153214918
Name: revenue, dtype: Int64

In [53]:
df["price_uzs"].value_counts()

price_uzs
0          10
-150000     3
85532       2
101004      2
459478      2
           ..
443014      1
164075      1
463372      1
86315       1
87538       1
Name: count, Length: 2470, dtype: Int64

In [54]:
df = df[df["price_uzs"] >= 0]

In [55]:
df["price_uzs"].value_counts()

price_uzs
0         10
85532      2
101004     2
459478     2
144494     2
          ..
443014     1
164075     1
463372     1
86315      1
87538      1
Name: count, Length: 2468, dtype: Int64

In [56]:
df.sample(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating,revenue
2038,TK-12366,pakhtakor vs bunyodkor,futbol,milliy stadion,2026-04-05,2026-03-29,xorazm,vip,452428,1,karta,qaytarilgan,NaN,452428
1566,TK-11831,pakhtakor vs bunyodkor,futbol,milliy stadion,05.04.2026,2026-04-03,farg'ona,standart,146546,3,karta,yakunlangan,4.7,439638
749,TK-11221,doniyor otajonov,konsert,uzbekiston konsert zali,2026-05-03,2026-05-01,andijon,fan,72981,1,qr,yakunlangan,4.1,72981
647,TK-11289,jasur umirov,konsert,uzbekiston konsert zali,12.04.2026,2026-03-19,buxoro,fan,73908,2,karta,yakunlangan,3.6,147816
2320,TK-11327,muhabbat shamayeva,konsert,uzbekiston konsert zali,17.05.2026,2026-04-27,buxoro,standart,149374,4,qr,yakunlangan,4.4,597496
1639,TK-10005,ulugbek rahmatullayev,konsert,humo arena,2026-03-22,2026-03-12,namangan,vip,454890,1,naqd,yakunlangan,4.6,454890
1765,TK-11988,pakhtakor vs lokomotiv,futbol,milliy stadion,10.05.2026,2026-03-26,farg'ona,fan,108662,3,naqd,yakunlangan,4.6,325986
735,TK-10444,navruz gala shou,konsert,humo arena,2026-03-21,2026-02-17,namangan,vip,458387,3,karta,yakunlangan,3.6,1375161
746,TK-10306,qxl stand-up show,stendap,hysteria club,26.04.2026,2026-03-20,namangan,standart,151498,4,naqd,yakunlangan,4.7,605992
962,TK-10305,ulugbek rahmatullayev,konsert,humo arena,2026-03-22,2026-03-07,namangan,fan,109961,4,naqd,yakunlangan,4.9,439844


In [57]:
df = df[~((df["price_uzs"] == 0) & (df["status"] != "yakunlangan"))]

In [58]:
df["price_uzs"].value_counts()

price_uzs
0         9
85532     2
101004    2
459478    2
144494    2
         ..
443014    1
164075    1
463372    1
86315     1
87538     1
Name: count, Length: 2468, dtype: Int64

In [59]:
df.describe()

,price_uzs,quantity,rating,revenue
count,2495.0,2495.000000,1897.000000,2495.0
mean,231403.127856,2.428858,4.245229,558014.361122
std,161784.545147,1.251963,0.441516,525827.691516
min,0.0,-4.000000,3.500000,-1864436.0
25%,97424.5,1.000000,3.900000,196813.0
50%,153597.0,3.000000,4.300000,407115.0
75%,443907.0,3.000000,4.600000,701166.0
max,479999.0,4.000000,5.000000,1919256.0


In [60]:
df.sample(10)

,ticket_id,artist_event,event_type,venue,event_date,purchase_date,customer_region,ticket_type,price_uzs,quantity,payment,status,rating,revenue
887,TK-10583,shaxlo ismoilova,konsert,humo arena,19.04.2026,2026-03-01,namangan,standart,156560,4,karta,yakunlangan,4.8,626240
2061,TK-11388,jasur umirov,konsert,uzbekiston konsert zali,12.04.2026,2026-02-15,farg'ona,standart,152816,4,naqd,qaytarilgan,NaN,611264
539,TK-11418,navruz gala shou,konsert,humo arena,21.03.2026,2026-02-13,toshkent,vip,462656,3,naqd,yakunlangan,3.6,1387968
1248,TK-10454,jasur umirov,konsert,uzbekiston konsert zali,2026-04-12,2026-02-12,qashqadaryo,fan,77960,3,karta,yakunlangan,5.0,233880
1192,TK-10338,jasur umirov,konsert,uzbekiston konsert zali,12.04.2026,2026-04-01,farg'ona,vip,456106,4,qr,yakunlangan,4.0,1824424
2479,TK-10161,qxl stand-up show,stendap,hysteria club,2026-04-26,2026-03-03,andijon,vip,476974,1,qr,yakunlangan,4.5,476974
2114,TK-11516,pakhtakor vs bunyodkor,futbol,milliy stadion,05.04.2026,2026-02-23,samarqand,vip,456247,3,karta,qaytarilgan,NaN,1368741
2409,TK-11597,sevinch mominova,konsert,humo arena,15.03.2026,2026-01-28,toshkent,vip,463313,3,qr,kutilmoqda,NaN,1389939
2245,TK-10127,ulugbek rahmatullayev,konsert,humo arena,2026-03-22,2026-02-06,farg'ona,standart,163751,2,naqd,qaytarilgan,NaN,327502
1207,TK-11535,muhabbat shamayeva,konsert,uzbekiston konsert zali,2026-05-17,2026-05-03,toshkent,vip,439912,3,qr,yakunlangan,5.0,1319736


In [61]:
df.to_csv("../data/concert_tickets_cleaned.csv", index=False)